# 11. Regresion UITI_VANO con M-GCECDL — corrida local con presupuesto de busqueda

Version **local** del experimento de la familia `uiti-vano-regression`, portada desde
`notebooks/kaggle_experiments/uiti-vano-regression-budget/notebook.ipynb` (corrida remota en
Kaggle) para que corra tal cual en esta maquina, sin `/kaggle/input` ni datasets adjuntos.

Reproduce la metodologia de `02.1_mgcecdl_regression_embeddings.ipynb` y la cierra contra el
agrupamiento por features crudas de `10_uiti_vano_kmeans.ipynb`.

**Reuso primero.** `MGCECDLRegressor`, `MGCECDLRegressionLoss` y `KernelDensityWeightedMSELoss`
se importan de `chec_impacto.models.mgcecdl`; `run_optuna_study`, `procesar_dataset_completo`,
`construir_modalidades_mgcecdl` y `construir_matriz_adyacencia_mgcecdl` se importan de sus
modulos. Este cuaderno **no redefine** ni el modelo ni la perdida: solo orquesta la corrida.

## Que cambia respecto de la version Kaggle

| Aspecto | Kaggle | Local (este cuaderno) |
|---|---|---|
| Codigo fuente | dataset privado `chec-impacto-src` montado en `/kaggle/input` | `src/` del checkout, agregado a `sys.path` |
| Datos | dataset privado `uiti-vano-indicadores-v3` | `data/Indicadores_vano_v3.csv` + `data/Variables_seleccion.xlsx` |
| Escritura | `/kaggle/working` (efimero) | `data/optuna/` y `reports/interpretability/figures/` del repo |
| Dispositivo | GPU del kernel | `resolve_training_device("auto")` → `mps:0` en Apple Silicon |
| Guarda del preflight | "reempaqueta el dataset" | "corre desde el checkout / revisa `src/`" |
| Presupuesto `full` | 15 trials (mayor que el baseline a proposito) | 10 trials = paridad exacta con el baseline local fijado |

## Linea base a batir

Fijada en `.claude/skills/experimento-kaggle/references/uiti-vano-regression-baseline.md`
(corrida local, `device=mps:0`, barrido de perdida @20 epochs, Optuna 10 trials @20 epochs,
reentrenamiento final @60 epochs):

| Metrica | Valor | Rol |
|---|---|---|
| `mae_original` | 126.402 | **Primaria** |
| `r2_original` | -0.027 | Diagnostico secundario |
| `r2_transformed` | 0.284 | Diagnostico secundario |
| `ARI` (K automatico) | 0.0000 | Diagnostico secundario |
| `ARI` (K=4 fijo) | 0.1115 | Diagnostico secundario |

**Artefactos:** este cuaderno escribe con nombres propios sufijados por `mode`
(`mgcecdl_regression_local_{mode}.journal` / `.pkl`) para **no sobrescribir** los artefactos
del baseline que dejo `02.1_` (`mgcecdl_regression_search.journal`,
`mgcecdl_regression_study.pkl`).

## Diagrama del proceso

```mermaid
flowchart TD
    A["data/Indicadores_vano_v3.csv<br/>+ Variables_seleccion.xlsx"] --> B["procesar_dataset_completo<br/>ventana climatica 12h, sin tope de UITI<br/>target UITI_VANO"]
    B --> C["X, y, features, df_identidad"]
    C --> D["construir_modalidades_mgcecdl(features)<br/>indices por modalidad"]
    C --> E["construir_matriz_adyacencia_mgcecdl<br/>grafo d x d entre predictores"]
    C --> F["Particion cronologica<br/>corte = percentil 80 de FECHA"]
    F --> G["MinMaxScaler en X<br/>log1p + MinMaxScaler en y<br/>(ajustados SOLO con train)"]
    G --> H["TensorDataset / DataLoader<br/>batch 512, shuffle solo en train"]
    D --> I
    E --> I
    H --> I["train_regressor()<br/>MGCECDLRegressor + MGCECDLRegressionLoss<br/>early stopping por valid_fused_loss"]
    I --> J["5. Barrido de forma de perdida<br/>mse / huber / kernel_weighted_mse<br/>selecciona por mae_original.idxmin()"]
    J --> K["6. Optuna GPSampler + MedianPruner<br/>objetivo: minimizar mae_original"]
    K --> L["7. Reentrenamiento final<br/>mejor forma de perdida + mejores hiperparametros"]
    L --> M["8. Embeddings de todos los eventos<br/>promediados por CIRCUITO + FID_VANO"]
    M --> N["9. K-Means K=2..8<br/>seleccion por silueta"]
    N --> O["10. Triangulacion ARI<br/>vs. K-Means de features crudas (Parte A)"]
    N --> Q["11. Proyeccion UMAP 2D<br/>color por cluster / UITI acumulado / n. eventos"]
    O --> Q
    L --> P["12. Reporte: mae_original vs. baseline 126.402"]
    O --> P
```

El orden **no es negociable**: barrido de perdida → Optuna → reentrenamiento final →
embeddings → K-Means/silueta → ARI. Cambiarlo rompe la comparabilidad con el baseline fijado.

## Funcion de costo

Las ecuaciones de esta seccion estan tomadas de la implementacion real
(`MGCECDLRegressionLoss.compute_components` y `_elementwise_regression_loss`, en
`src/chec_impacto/models/mgcecdl.py`), **no** de la descripcion generica de
`docs/mgcecdl_normalizacion_funcion_costo.md`. Ver la nota de discrepancia al final: para
regresion, ese doc describe una normalizacion que esta clase **deliberadamente no aplica**.

### Notacion

| Simbolo | Significado |
|---|---|
| $B$ | tamano del lote; $i = 1..B$ |
| $M$ | numero de modalidades; $d = 1..M$ |
| $y_i$ | objetivo en **escala transformada** (`log1p` → MinMax), en $[0,1]$ |
| $\hat{y}_i$ | prediccion fusionada (`fused_prediction`) |
| $\hat{y}_{i,d}$ | prediccion de la cabeza de la modalidad $d$ |
| $\pi_{i,d}$ | fiabilidad (`reliabilities`) de la modalidad $d$ para la muestra $i$, con $\sum_d \pi_{i,d}=1$ |
| $x_i \in \mathbb{R}^{p}$ | predictores escalados de entrada; $p$ = numero de features |
| $\hat{x}_i$ | reconstruccion de $x_i$ por los decoders (`reconstructed_features`) |
| $A \in \mathbb{R}^{p\times p}$ | matriz de adyacencia del grafo de predictores |

### Perdida total

$$
\mathcal{L}^{total}
= \mathcal{L}_{fused}
+ \gamma_{sup}\,\mathcal{L}_{modality}
+ \gamma_{agr}\,\mathcal{L}_{agreement}
+ \gamma_{reg}\,\widetilde{\mathcal{L}}_{KL}
+ \lambda_{rec}\,\widetilde{\mathcal{L}}_{rec}
+ \lambda_{MI}\,\widetilde{\mathcal{L}}_{MI}
$$

La tilde marca **solo** los terminos que si estan acotados a $[0,1]$.

### Los tres primeros terminos (sin acotar)

Sea $\ell(\cdot,\cdot)$ la **perdida base elemento a elemento** — la que se barre en la
seccion 5 y se define abajo:

$$
\mathcal{L}_{fused} = \frac{1}{B}\sum_{i=1}^{B} \ell(\hat{y}_i,\, y_i)
$$

$$
\mathcal{L}_{modality} = \frac{1}{B}\sum_{i=1}^{B}\ \sum_{d=1}^{M} \pi_{i,d}\ \ell(\hat{y}_{i,d},\, y_i)
$$

(supervision por rama, **ponderada por fiabilidad**; con
`weight_modality_loss_by_reliability=False` seria la media simple sobre las modalidades activas)

$$
\mathcal{L}_{agreement} = \frac{1}{B}\sum_{i=1}^{B}\ \sum_{d=1}^{M} \pi_{i,d}\,\bigl(\hat{y}_{i,d} - \hat{y}_i\bigr)^{2}
$$

(desviacion cuadratica de cada modalidad respecto del consenso, ponderada por fiabilidad)

### Los tres terminos acotados

$$
\widetilde{\mathcal{L}}_{KL} = \operatorname{clip}\!\left(\frac{D_{KL}\!\left(u \,\|\, \pi\right)}{\log \max(M,2)},\, 0,\, 1\right),
\qquad
D_{KL}(u\,\|\,\pi) = \frac{1}{B}\sum_{i=1}^{B}\sum_{d=1}^{M} \frac{1}{M}\log\frac{1/M}{\pi_{i,d}}
$$

Ojo con el orden de los argumentos: `F.kl_div(log π, uniform, reduction="batchmean")` calcula
$D_{KL}(u\,\|\,\pi)$, la divergencia de la **uniforme respecto de las fiabilidades** — penaliza
que el gating colapse en una sola modalidad.

$$
\widetilde{\mathcal{L}}_{rec} = \operatorname{clip}\!\left(\frac{1}{B\,p}\sum_{i,j}\bigl(\hat{x}_{i,j} - z_{i,j}\bigr)^{2},\, 0,\, 1\right),
\qquad
z_{i,j} = \frac{x_{i,j} - \mu_j}{\sigma_j}
$$

con $\mu_j,\sigma_j$ calculados **solo con train** (`calcular_estadisticas_reconstruccion_mgcecdl`).

$$
\widetilde{\mathcal{L}}_{MI} = 1 - \operatorname{clip}\!\left(\frac{I_2\!\left(K_{rec},\, K_{graph}\right)}{\log \max(p,2)},\, 0,\, 1\right)
$$

$I_2$ es la informacion mutua de **Renyi cuadratica (orden 2)**, calculada con entropias de
Renyi de orden 2 sobre kernels, sin estimar densidades:

$$
I_2(K_{rec}, K_{graph}) = H_2(K_{rec}) + H_2(K_{graph}) - H_2(K_{rec} \odot K_{graph})
$$

$$
H_2(K) = -\log \sum_{a,b} \left(\frac{K_{ab}}{\operatorname{tr} K}\right)^{2}
$$

donde $\odot$ es el producto de Hadamard (kernel conjunto). $K_{rec}$ es el kernel RBF de los
**perfiles de variable** reconstruidos ($\hat{X}^{\top}$, es decir cada predictor descrito por
como se reconstruye a lo largo del lote) y $K_{graph}$ el kernel RBF de los perfiles del grafo
$[A \,\|\, A^{\top}]$, con $\sigma_{graph}$ = **mediana de las distancias euclidianas por pares**
entre esos perfiles (heuristica de la mediana) y $\sigma_{rec}$ = `rbf_sigma` = $1.0$.

Maximizar informacion mutua equivale a minimizar esta perdida; $0$ es el mejor caso, y
significa maxima alineacion entre la estructura que el modelo reconstruye y la del grafo.

---

## Las tres perdidas base probadas ($\ell$)

Es lo unico que cambia en el barrido de la seccion 5: misma arquitectura, mismo presupuesto de
epochs, mismos $\gamma$/$\lambda$ por defecto para las tres candidatas.

### 1. `mse`

$$
\ell_{MSE}(\hat{y}, y) = (\hat{y} - y)^{2}
$$

Referencia neutra. Penaliza cuadraticamente, asi que la cola de `UITI_VANO` (muy larga incluso
despues de `log1p`) domina el gradiente.

### 2. `huber`

`F.huber_loss(..., reduction="none", delta=δ)` con $\delta = 1.0$ (`huber_delta`, valor por
defecto que este cuaderno no altera):

$$
\ell_{\delta}(\hat{y}, y) =
\begin{cases}
\dfrac{1}{2}\,(\hat{y} - y)^{2}, & |\hat{y} - y| \le \delta \\[2ex]
\delta\left(|\hat{y} - y| - \dfrac{1}{2}\delta\right), & |\hat{y} - y| > \delta
\end{cases}
$$

Cuadratica cerca de cero y **lineal** en la cola: el gradiente se satura en $\delta$, asi que un
residuo enorme no puede secuestrar la actualizacion. Es la version de PyTorch escalada por
$\delta$ (no `smooth_l1_loss`, que difiere en un factor $\delta$).

Nota de escala: como $y$ vive en $[0,1]$ tras `log1p`+MinMax, casi todos los residuos cumplen
$|\hat{y}-y| \le 1 = \delta$, asi que en la practica `huber` opera casi siempre en su rama
cuadratica y equivale a $\tfrac{1}{2}\ell_{MSE}$ — **la mitad de la escala de `mse`**. Esto
explica por que su `fused_loss` sale sistematicamente mas baja que la de `mse` sin que eso
signifique, por si solo, que prediga mejor. Por eso el ganador se elige por `mae_original`, no
por la perdida.

### 3. `kernel_weighted_mse`

$$
\ell_{KW}(\hat{y}, y) = w(y)\,(\hat{y} - y)^{2},
\qquad
w(y) = \frac{1}{\max\bigl(\hat{p}(y),\, \varepsilon\bigr)},\quad \varepsilon = 10^{-6}
$$

donde $\hat{p}$ es la densidad del objetivo estimada por KDE gaussiana sobre `y_train`
(`gaussian_kde`, ancho de banda por regla de Scott), evaluada una sola vez sobre una malla fija
$\{v_1 < \dots < v_K\}$, $K = 512$, y despues **interpolada linealmente**:

$$
\hat{p}(y) = (1-t)\,\hat{p}(v_k) + t\,\hat{p}(v_{k+1}),
\qquad
t = \frac{\operatorname{clip}(y, v_1, v_K) - v_k}{v_{k+1} - v_k}
$$

La KDE se ajusta **una sola vez** y por lote solo se interpola, para no pagar
$O(B \times n_{train})$ evaluaciones de kernel en cada paso.

Intencion: subir el peso de las regiones **raras** del objetivo — exactamente donde viven los
vanos criticos y donde `mse` los ahoga bajo la masa densa del rango medio.

> **Detalle de escala que importa.** La clase `KernelDensityWeightedMSELoss.forward` normaliza
> los pesos a media 1 por lote ($w \leftarrow w/\bar{w}$), pero dentro de la perdida compuesta
> se llama a `compute_weights`, que devuelve el **inverso crudo de la densidad, sin
> renormalizar**. Es decir: dentro de `MGCECDLRegressionLoss`, `kernel_weighted_mse` entra en
> una escala distinta a `mse`/`huber`. Por eso su `fused_loss` es varias veces mayor en el
> barrido — y por eso, otra vez, la seleccion se hace por `mae_original` y no comparando
> perdidas entre formas distintas.

El ganador se elige por `mae_original.idxmin()` — MAE en la **escala real** de `UITI_VANO`
(tras invertir MinMax y `expm1`), no en la escala transformada — y es el que se usa en Optuna y
en el reentrenamiento final.

---

### Por que `fused`/`modality`/`agreement` NO se acotan (discrepancia documentada)

`docs/mgcecdl_normalizacion_funcion_costo.md` describe, para regresion, una normalizacion de
`fused`/`modality` por la Huber de un predictor constante y de `disagreement` por
$\operatorname{var}(y_{train})$, todo con `clip(·,0,1)`. **`MGCECDLRegressionLoss` no hace
nada de eso**, y su docstring explica por que:

> los residuos de regresion no estan acotados de forma natural a `[0,1]`; acotarlos igualaria
> en silencio formas de perdida muy distintas (p. ej. MSE vs. Huber sobre residuos grandes) y
> anularia justo la comparacion empirica de formas de perdida para la que existe esta clase.

Solo `KL`, `rec` y `MI` estan acotados, porque esos si tienen una escala de referencia natural
($\log M$, escala estandarizada, $\log p$). Consecuencia practica: los rangos que Optuna
explora para `gamma_agr` y `gamma_sup` **no son directamente comparables** con los de
clasificacion, donde todos los terminos si comparten escala.

**Nota sobre lo que aporta el grafo.** `agreement`, `KL`, `rec` y `MI` son los terminos que
distinguen a M-GCECDL de una MLP multimodal cualquiera: `rec` y `MI` obligan a que los
embeddings sigan explicando los predictores **y** su estructura de grafo, no solo el target.

## Particion de datos

**Cronologica, no aleatoria.** El corte es el **percentil 80 de `FECHA`**:

```python
FECHA_CORTE = df_identidad["FECHA"].quantile(0.8)
train_mask = fechas <= FECHA_CORTE
valid_mask = ~train_mask
```

Un split aleatorio sobre eventos del mismo vano y la misma ventana climatica filtraria
informacion del futuro al entrenamiento e inflaria las metricas. El corte temporal fuerza la
pregunta util: *con lo observado hasta cierta fecha, que tan bien se predice el UITI de los
eventos posteriores*.

**Escalado ajustado solo con train** (`fit` en train, `transform` en valid y en el conjunto
completo), tanto para `X` (`MinMaxScaler`) como para `y` (`log1p` → `MinMaxScaler`).
`log1p` antes del MinMax comprime la cola larga de `UITI_VANO`; las metricas en escala real se
recuperan invirtiendo la cadena (`y_scaler.inverse_transform` → `expm1`).

**Sin conjunto de test aparte.** Optuna selecciona sobre el mismo `valid`, asi que
`mae_original` reportado es optimista respecto de datos verdaderamente nuevos. Es la misma
condicion del baseline fijado, por lo que la **comparacion** contra `126.402` sigue siendo
valida; el numero absoluto no debe leerse como desempeno en produccion.

**Agregacion para clustering.** Los embeddings se calculan por evento sobre *todos* los datos
(train + valid) y se promedian por `CIRCUITO` + `FID_VANO`, porque la unidad de analisis del
agrupamiento es el vano, no el evento.

## Entrenamiento y busqueda de hiperparametros

**Bucle de entrenamiento** (`train_regressor`, seccion 4): early stopping por
`valid_fused_loss` con `patience`, checkpoint del mejor epoch restaurado antes de medir, y un
`assert` que aborta si el modelo aterriza en un dispositivo distinto al solicitado — para no
entrenar horas en CPU creyendo que se usa la GPU.

**Optuna** (seccion 6): `GPSampler` (modelo sustituto gaussiano) + `MedianPruner`
(`n_startup_trials=3`), reutilizados via `run_optuna_study`, con almacenamiento en journal
(reanudable). Objetivo: **minimizar `mae_original`**.

Espacio de busqueda:

| Grupo | Parametro | Rango |
|---|---|---|
| Arquitectura | `hidden_dim` | {64, 128, 192} |
| | `embed_dim` | {32, 64, 96} |
| | `dropout` | [0.0, 0.25] |
| Optimizacion | `optimizer_type` | {adam, adamw, sgd, rmsprop} |
| | `learning_rate` | [1e-4, 1e-2] log |
| | `weight_decay` | [1e-6, 1e-4] log |
| | `momentum` | [0.5, 0.95] (solo sgd/rmsprop) |
| | `batch_size` | {256, 512, 1024} |
| Pesos de la perdida | `gamma_sup`, `gamma_agr`, `gamma_reg` | [1e-2, 1.0] log cada uno |
| | `lambda_reconstruction`, `lambda_mutual_information` | [1e-2, 1.0] log cada uno |

La forma de la perdida base **no** entra al espacio de Optuna: ya quedo fijada por el barrido
de la seccion 5. `rbf_sigma` se mantiene en `1.0` (paridad con el baseline), aunque
`docs/mgcecdl_normalizacion_funcion_costo.md` lo lista como sintonizable — ampliarlo cambiaria
el espacio respecto del baseline fijado.

**Presupuesto.** `mode="smoke"` es una prueba de cableado de menos de un minuto (submuestra de
2000 eventos, 2 trials, 2-3 epochs) — **sus numeros no significan nada**. `mode="full"` corre
la paridad exacta del baseline: barrido @20 epochs, 10 trials @20 epochs, reentrenamiento final
@60 epochs. En Apple Silicon (`mps:0`, ~3.2 s/epoch en el baseline) esto es del orden de
**decenas de minutos**, no de segundos.

In [ ]:
# Celda de parametros (papermill). Sobrescribir con `-p mode full` para la corrida real.
# El default es "smoke" a proposito: una ejecucion desprevenida no debe lanzar el
# presupuesto completo por accidente.
mode = "smoke"

## Bootstrap: raiz del repo, `sys.path` y guarda de precondiciones

La guarda de precondiciones es obligatoria para esta familia de experimentos (ver
`references/uiti-vano-regression-baseline.md`): debe fallar rapido y con un mensaje accionable
si el modelo/perdida no son importables, en vez de que el cuaderno improvise una
implementacion sustituta mas abajo.

In [ ]:
import sys
from pathlib import Path


# Sube desde el cwd hasta el checkout, para correr desde cualquier directorio.
def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src" / "chec_impacto").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError(
        "No se encontro la raiz del proyecto (se busco un directorio con src/chec_impacto/ "
        "y data/ subiendo desde el cwd). Ejecuta este cuaderno desde el checkout."
    )


PROJECT_ROOT = resolve_project_root()
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
FIGURES_DIR = PROJECT_ROOT / "reports" / "interpretability" / "figures"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:    ", DATA_DIR)
print("FIGURES_DIR: ", FIGURES_DIR)

# --- Guarda de precondiciones: sonda de import, nunca una implementacion sustituta ---
try:
    from chec_impacto.models.mgcecdl import (
        MGCECDLRegressor,
        MGCECDLRegressionLoss,
        KernelDensityWeightedMSELoss,
    )
except ImportError as exc:
    raise SystemExit(
        "MGCECDLRegressor/MGCECDLRegressionLoss no son importables. Estas clases viven en "
        f"src/chec_impacto/models/mgcecdl.py; se busco en {SRC_DIR}. Verifica que corres "
        "desde el checkout y que el entorno tiene torch/optuna instalados "
        "(pip install -r requirements.txt)."
    ) from exc

print("Guarda OK: MGCECDLRegressor / MGCECDLRegressionLoss / KernelDensityWeightedMSELoss importables.")

In [ ]:
import json
import time as _time

import numpy as np
import pandas as pd
import torch
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, mean_absolute_error, r2_score, silhouette_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

from chec_impacto.data import construir_matriz_adyacencia_mgcecdl, procesar_dataset_completo
from chec_impacto.training import (
    calcular_estadisticas_reconstruccion_mgcecdl,
    construir_modalidades_mgcecdl,
    guardar_estudio_optuna,
    resolve_training_device,
)
from chec_impacto.training.mgcecdl import run_optuna_study
import optuna

RANDOM_STATE = 42

# resolve_training_device("auto") prioriza CUDA -> MPS -> CPU (src/chec_impacto/training/
# mgcecdl.py). Para CUDA no se conforma con is_available(): ejecuta un probe real y cae a CPU
# con RuntimeWarning si el driver esta presente pero no puede lanzar kernels. Aqui se imprime
# el diagnostico completo para que quede constancia de en que hardware corrio el experimento.
DEVICE = resolve_training_device("auto")
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("  GPU:", torch.cuda.get_device_name(0))
print("MPS (Metal Performance Shaders) disponible:", torch.backends.mps.is_available())
print(f"Dispositivo de entrenamiento resuelto: {DEVICE}")
if DEVICE.type == "cpu":
    print(
        "AVISO: se entrenara en CPU. Con mode='full' esto puede tardar horas en vez de "
        "minutos; el baseline fijado se midio en mps:0 (~3.2 s/epoch)."
    )

## Configuracion del presupuesto (`smoke` vs `full`)

In [ ]:
if mode not in ("smoke", "full"):
    raise ValueError(f"mode desconocido: {mode!r} -- se esperaba 'smoke' o 'full'.")

if mode == "smoke":
    # Prueba de cableado: submuestra pequena y presupuesto minimo. Termina en menos de
    # un minuto. Sus metricas NO son comparables con el baseline.
    SMOKE_EVENT_SUBSAMPLE = 2000
    LOSS_COMPARISON_MAX_EPOCHS = 2
    LOSS_COMPARISON_PATIENCE = 2
    OPTUNA_N_TRIALS = 2
    OPTUNA_MAX_EPOCHS = 2
    OPTUNA_PATIENCE = 2
    FINAL_MAX_EPOCHS = 3
    FINAL_PATIENCE = 3
else:
    # Paridad exacta con el baseline local fijado en
    # references/uiti-vano-regression-baseline.md: mismo presupuesto, misma metodologia,
    # mismo hardware => el delta de mae_original es atribuible a la corrida, no al budget.
    # (La variante Kaggle sube OPTUNA_N_TRIALS a 15 a proposito; aqui no, para conservar
    # la comparabilidad directa contra 126.402.)
    SMOKE_EVENT_SUBSAMPLE = None
    LOSS_COMPARISON_MAX_EPOCHS = 20
    LOSS_COMPARISON_PATIENCE = 7
    OPTUNA_N_TRIALS = 10
    OPTUNA_MAX_EPOCHS = 20
    OPTUNA_PATIENCE = 7
    FINAL_MAX_EPOCHS = 60
    FINAL_PATIENCE = 15

print(f"mode={mode!r}")
print(
    f"barrido_epochs={LOSS_COMPARISON_MAX_EPOCHS} | optuna_trials={OPTUNA_N_TRIALS} | "
    f"optuna_epochs={OPTUNA_MAX_EPOCHS} | final_epochs={FINAL_MAX_EPOCHS} | "
    f"submuestra={SMOKE_EVENT_SUBSAMPLE}"
)

## 1. Carga de datos

Misma llamada que `02.1_`: `procesar_dataset_completo` (transformacion `log1p` + MinMax del
target, ventana climatica de 12h, sin tope de UITI). El grafo de adyacencia se **reconstruye en
memoria** con `construir_matriz_adyacencia_mgcecdl` en vez de leerse de `data/graphs/`: ese
directorio es una cache opcional de reconstruccion y reconstruirlo garantiza que el grafo
corresponde exactamente a las `features` de esta corrida.

In [ ]:
VENTANA_CLIMATICA_HORAS = 12
FILTRO_UITI_MAX = None

DATASET_PATH = DATA_DIR / "Indicadores_vano_v3.csv"
VARIABLES_SELECCION_PATH = DATA_DIR / "Variables_seleccion.xlsx"
for required_path in (DATASET_PATH, VARIABLES_SELECCION_PATH):
    if not required_path.exists():
        raise FileNotFoundError(f"Falta {required_path} -- revisa el contenido de data/.")

datos_procesados = procesar_dataset_completo(
    path_clima=DATASET_PATH,
    path_variables_seleccion=VARIABLES_SELECCION_PATH,
    use_sampling=False,
    min_samples_per_codigo=5,
    target="UITI_VANO",
    filtro_uiti_max=FILTRO_UITI_MAX,
    ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
)

X = datos_procesados["X"]
y = datos_procesados["y"]
features = datos_procesados["features"]
df_identidad = datos_procesados["df_original_copy"].reset_index(drop=True)
assert len(df_identidad) == len(X) == len(y)

if SMOKE_EVENT_SUBSAMPLE is not None and len(df_identidad) > SMOKE_EVENT_SUBSAMPLE:
    rng = np.random.RandomState(RANDOM_STATE)
    subsample_idx = np.sort(rng.choice(len(df_identidad), size=SMOKE_EVENT_SUBSAMPLE, replace=False))
    X = X[subsample_idx]
    y = y[subsample_idx]
    df_identidad = df_identidad.iloc[subsample_idx].reset_index(drop=True)

modality_feature_indices = construir_modalidades_mgcecdl(features)
graph_adjacency_matrix, _edges = construir_matriz_adyacencia_mgcecdl(
    features, ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
)
assert graph_adjacency_matrix.shape == (len(features), len(features))

print("X:", X.shape, "| y:", y.shape)
print("Modalidades:", {name: len(indices) for name, indices in modality_feature_indices.items()})
print("Matriz de adyacencia:", graph_adjacency_matrix.shape)

## 2. Particion cronologica train/valid (corte por percentil 80 de `FECHA`)

In [ ]:
fechas = df_identidad["FECHA"]
FECHA_CORTE = fechas.quantile(0.8)
train_mask = (fechas <= FECHA_CORTE).to_numpy()
valid_mask = ~train_mask

print(f"Fecha de corte (percentil 80): {FECHA_CORTE}")
print(f"Train: {train_mask.sum()} eventos | Valid: {valid_mask.sum()} eventos")

## 3. Escalado, tensores y estadisticas de reconstruccion

Los `scaler` se ajustan **solo con train**; `valid` y el conjunto completo solo se transforman.
`calcular_estadisticas_reconstruccion_mgcecdl` produce la media/desviacion de train que la
perdida usa para medir la reconstruccion en escala estandarizada.

In [ ]:
x_scaler = MinMaxScaler()
X_train_scaled = x_scaler.fit_transform(X[train_mask]).astype(np.float32)
X_valid_scaled = x_scaler.transform(X[valid_mask]).astype(np.float32)
X_full_scaled = x_scaler.transform(X).astype(np.float32)

y_log1p_all = np.log1p(y)
y_scaler = MinMaxScaler()
y_train_scaled = y_scaler.fit_transform(y_log1p_all[train_mask]).astype(np.float32).reshape(-1)
y_valid_scaled = y_scaler.transform(y_log1p_all[valid_mask]).astype(np.float32).reshape(-1)

feature_mean, feature_std = calcular_estadisticas_reconstruccion_mgcecdl(X_train_scaled)

BATCH_SIZE = 512
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


def make_loaders(batch_size=BATCH_SIZE, seed=RANDOM_STATE):
    train_dataset = TensorDataset(torch.tensor(X_train_scaled), torch.tensor(y_train_scaled))
    valid_dataset = TensorDataset(torch.tensor(X_valid_scaled), torch.tensor(y_valid_scaled))
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )
    valid_loader = DataLoader(valid_dataset, batch_size=1024, shuffle=False)
    return train_loader, valid_loader


train_loader, valid_loader = make_loaders()
print("Batches train:", len(train_loader), "| batches valid:", len(valid_loader))

## 4. Bucle de entrenamiento compartido (`train_regressor`)

Misma logica que `02.1_` y que la version Kaggle: misma composicion de la perdida
(`loss_fn.compute_components`), mismo `assert` de aterrizaje en dispositivo (falla ruidosamente
en vez de entrenar en CPU en silencio cuando se pidio GPU/MPS), seleccion del mejor checkpoint
por `valid_fused_loss`, y el mismo conjunto de metricas (`mae_original`, `r2_original`,
`r2_transformed`), **mas** las versiones ajustadas (`r2_original_adj`, `r2_transformed_adj`) y
el $n$/$p$ con que se calcularon — ver la celda de metricas de abajo.

### Metricas de bondad de ajuste: $R^2$ y $R^2$ ajustado

**$R^2$ (coeficiente de determinacion).** Fraccion de la varianza del objetivo que el modelo
explica, relativa a predecir siempre la media:

$$
R^{2}
= 1 - \frac{SS_{res}}{SS_{tot}}
= 1 - \frac{\sum_{i=1}^{n}\left(y_i - \hat{y}_i\right)^{2}}{\sum_{i=1}^{n}\left(y_i - \bar{y}\right)^{2}},
\qquad
\bar{y} = \frac{1}{n}\sum_{i=1}^{n} y_i
$$

$R^2 = 1$ es prediccion perfecta y $R^2 = 0$ equivale a predecir siempre $\bar{y}$. **Puede ser
negativo**: significa que el modelo lo hace *peor* que la media constante — que es exactamente
el caso del baseline fijado ($R^2_{original} = -0.027$).

**$R^2$ ajustado.** El $R^2$ nunca baja al agregar predictores, aunque sean ruido puro: cada
variable nueva siempre absorbe algo de varianza. El ajustado descuenta ese regalo penalizando
por el numero de predictores $p$ frente al numero de observaciones $n$:

$$
R^{2}_{adj} = 1 - \left(1 - R^{2}\right)\,\frac{n - 1}{n - p - 1}
$$

En este experimento $p = 70$ (las columnas de `features` que entran al modelo) y $n$ es el
numero de eventos de **validacion** sobre los que se mide.

Como $\frac{n-1}{n-p-1} \ge 1$, siempre se cumple $R^{2}_{adj} \le R^{2}$, y la brecha crece
cuando $p$ se acerca a $n$. Si $R^2$ ya es negativo, el ajustado lo es aun mas.

#### Dos advertencias honestas sobre leer el ajustado aqui

1. **El factor es casi irrelevante con este $n$.** En `mode='full'`, $n \approx 31{,}900$ y
   $p = 70$, asi que $\frac{n-1}{n-p-1} \approx 1.0022$: el ajuste mueve el tercer decimal.
   Solo en `mode='smoke'` ($n = 400$) el factor sube a $\approx 1.21$ y se nota. **El ajustado
   no va a rescatar un $R^2$ malo**; si el modelo explica poco, ambos lo dicen igual.
2. **$p$ aqui no son los grados de libertad reales del modelo.** El $R^2$ ajustado viene de la
   regresion lineal, donde $p$ coeficientes son exactamente la capacidad del modelo. Un
   `MGCECDLRegressor` tiene decenas de miles de parametros, no 70; y ademas estas metricas se
   miden **fuera de muestra** (en validacion), donde la penalizacion por capacidad ya la aplica
   el propio conjunto de validacion. Tomalo como un descuento conservador de cortesia, no como
   una correccion rigurosa de grados de libertad.

Por eso el cuaderno reporta **ambos**: el ajustado como diagnostico pedido, y el $R^2$ plano
porque es el que esta fijado en el baseline y el unico que permite una comparacion
directa contra `126.402` y compania. Nada de esto desplaza a `mae_original`, que sigue siendo
la metrica primaria de exito.

In [ ]:
# Terminos que se registran por epoch. Son exactamente los que suman el total:
#   total = fused + g_sup*modality + g_agr*agreement + g_reg*regularization
#           + l_rec*reconstruction + l_MI*mutual_information
TRACKED_LOSS_KEYS = (
    "total_loss",
    "fused_loss",
    "modality_loss",
    "agreement_loss",
    "regularization_loss",
    "reconstruction_loss",
    "mutual_information_loss",
)


def format_duration(seconds):
    seconds = int(max(seconds, 0))
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    return f"{hours:d}:{minutes:02d}:{secs:02d}" if hours else f"{minutes:02d}:{secs:02d}"


def loss_term_weights(loss_fn):
    # Los pesos con que cada termino entra al total. Se leen del propio objeto de perdida
    # (no se re-declaran) para que el desglose siga siendo correcto cuando Optuna los cambia.
    graph_term = loss_fn._graph_reconstruction
    return {
        "fused_loss": 1.0,
        "modality_loss": loss_fn.gamma_sup,
        "agreement_loss": loss_fn.gamma_agr,
        "regularization_loss": loss_fn.gamma_reg,
        "reconstruction_loss": graph_term.lambda_reconstruction,
        "mutual_information_loss": graph_term.lambda_mutual_information,
    }


def adjusted_r2(r2_value, n_samples, n_predictors):
    # R2_adj = 1 - (1 - R2) * (n - 1) / (n - p - 1). Ver la celda anterior para la derivacion
    # y sus dos advertencias de lectura.
    denominator = n_samples - n_predictors - 1
    if denominator <= 0:
        # Con p >= n - 1 el ajuste no esta definido; devolver nan es mas honesto que un numero
        # inventado o que reciclar el R2 plano haciendolo pasar por ajustado.
        return float("nan")
    return 1.0 - (1.0 - r2_value) * (n_samples - 1) / denominator


def build_regression_loss(base_loss, gamma_sup=0.20, gamma_agr=0.10, gamma_reg=0.01, kernel_loss_module=None,
                          lambda_reconstruction=0.01, lambda_mutual_information=0.01):
    return MGCECDLRegressionLoss(
        base_loss=base_loss,
        gamma_sup=gamma_sup,
        gamma_agr=gamma_agr,
        gamma_reg=gamma_reg,
        kernel_loss_module=kernel_loss_module,
        feature_mean=feature_mean,
        feature_std=feature_std,
        adjacency_matrix=graph_adjacency_matrix,
        rbf_sigma=1.0,
        lambda_reconstruction=lambda_reconstruction,
        lambda_mutual_information=lambda_mutual_information,
    )


def train_regressor(
    loss_fn,
    hidden_dim=128,
    embed_dim=64,
    dropout=0.10,
    lr=1e-3,
    weight_decay=1e-5,
    optimizer_type="adamw",
    momentum=0.0,
    batch_size=BATCH_SIZE,
    max_epochs=60,
    patience=15,
    seed=RANDOM_STATE,
    verbose=False,
):
    torch.manual_seed(seed)
    np.random.seed(seed)
    local_train_loader, local_valid_loader = make_loaders(batch_size=batch_size, seed=seed)

    model = MGCECDLRegressor(
        modality_feature_indices=modality_feature_indices,
        hidden_dim=hidden_dim, embed_dim=embed_dim, dropout=dropout,
    ).to(DEVICE)
    loss_fn = loss_fn.to(DEVICE)
    actual_param_device = next(model.parameters()).device
    assert actual_param_device.type == torch.device(DEVICE).type, (
        f"El modelo aterrizo en {actual_param_device} pero se pidio DEVICE={DEVICE} -- "
        "fallback silencioso, se aborta en vez de entrenar en el dispositivo equivocado."
    )

    optimizer_type = optimizer_type.lower()
    if optimizer_type == "adamw":
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_type == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_type == "sgd":
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    elif optimizer_type == "rmsprop":
        optimizer = torch.optim.RMSprop(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    else:
        raise ValueError(f"Optimizador no soportado: {optimizer_type}")

    def run_epoch(loader, train):
        model.train(mode=train)
        component_sums = np.zeros(len(TRACKED_LOSS_KEYS), dtype=np.float64)
        preds, targets_list = [], []
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if train:
                optimizer.zero_grad()
            with torch.set_grad_enabled(train):
                out = model(xb)
                components = loss_fn.compute_components(out, yb, xb)
                if train:
                    components["total_loss"].backward()
                    optimizer.step()
            # Un solo stack + .cpu() por lote en vez de un float() por termino: cada float()
            # fuerza una sincronizacion con GPU/MPS y siete de ellas por lote se notan.
            component_sums += torch.stack(
                [components[key].detach() for key in TRACKED_LOSS_KEYS]
            ).cpu().numpy().astype(np.float64)
            preds.append(out["fused_prediction"].detach().cpu().numpy())
            targets_list.append(yb.detach().cpu().numpy())
        epoch_components = dict(zip(TRACKED_LOSS_KEYS, component_sums / max(len(loader), 1)))
        return epoch_components, np.concatenate(preds), np.concatenate(targets_list)

    history = []
    best_valid_fused = float("inf")
    best_epoch = -1
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, max_epochs + 1):
        _epoch_t0 = _time.time()
        train_components, _, _ = run_epoch(local_train_loader, train=True)
        valid_components, _, _ = run_epoch(local_valid_loader, train=False)
        _epoch_elapsed = _time.time() - _epoch_t0
        valid_fused = valid_components["fused_loss"]

        history.append({
            "epoch": epoch,
            "epoch_seconds": _epoch_elapsed,
            **{f"train_{key}": value for key, value in train_components.items()},
            **{f"valid_{key}": value for key, value in valid_components.items()},
        })

        if valid_fused < best_valid_fused - 1e-7:
            best_valid_fused = valid_fused
            best_epoch = epoch
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if verbose:
            # Se imprime CADA epoch (no una de cada cinco): en mode='full' el reentrenamiento
            # dura decenas de minutos y hay que poder ver la evolucion mientras corre.
            # El ETA es una COTA SUPERIOR: asume llegar a max_epochs, y el early stopping
            # puede cortar antes.
            mean_epoch_seconds = sum(row["epoch_seconds"] for row in history) / len(history)
            eta_upper_bound = mean_epoch_seconds * (max_epochs - epoch)
            print(
                f"epoch {epoch:03d}/{max_epochs} | {_epoch_elapsed:5.2f}s | "
                f"train_total={train_components['total_loss']:.5f} "
                f"valid_total={valid_components['total_loss']:.5f} | "
                f"valid_fused={valid_fused:.5f} (mejor {best_valid_fused:.5f} @{best_epoch}) | "
                f"ETA <= {format_duration(eta_upper_bound)}"
            )

        if epochs_without_improvement >= patience:
            if verbose:
                print(f"Early stopping en epoch {epoch} "
                      f"(tiempo total: {format_duration(sum(row['epoch_seconds'] for row in history))}).")
            break

    if verbose and epochs_without_improvement < patience:
        print(f"Fin en epoch {len(history)}/{max_epochs} "
              f"(tiempo total: {format_duration(sum(row['epoch_seconds'] for row in history))}).")

    model.load_state_dict(best_state)
    model.eval()
    _, valid_preds_scaled, valid_targets_scaled = run_epoch(local_valid_loader, train=False)

    valid_preds_log1p = y_scaler.inverse_transform(valid_preds_scaled.reshape(-1, 1)).reshape(-1)
    valid_targets_log1p = y_scaler.inverse_transform(valid_targets_scaled.reshape(-1, 1)).reshape(-1)
    valid_preds_original = np.expm1(valid_preds_log1p)
    valid_targets_original = np.expm1(valid_targets_log1p)

    # n = eventos de validacion sobre los que se mide; p = predictores que entran al modelo.
    n_valid = len(valid_targets_original)
    n_predictors = len(features)
    r2_transformed = r2_score(valid_targets_scaled, valid_preds_scaled)
    r2_original = r2_score(valid_targets_original, valid_preds_original)

    metrics = {
        "best_epoch": best_epoch,
        "best_valid_fused_loss": best_valid_fused,
        "device": str(actual_param_device),
        "r2_transformed": r2_transformed,
        "r2_transformed_adj": adjusted_r2(r2_transformed, n_valid, n_predictors),
        "r2_original": r2_original,
        "r2_original_adj": adjusted_r2(r2_original, n_valid, n_predictors),
        "mae_original": mean_absolute_error(valid_targets_original, valid_preds_original),
        "n_valid": n_valid,
        "n_predictors": n_predictors,
    }
    return model, history, metrics

## 5. Barrido de forma de perdida (`mse` / `huber` / `kernel_weighted_mse`)

Misma arquitectura (`hidden_dim=128, embed_dim=64, dropout=0.10`) y mismo presupuesto de epochs
para las tres candidatas; la seleccion es por `mae_original.idxmin()`, calculada, no fijada a
mano.

Lo unico que cambia entre las tres corridas de abajo es $\ell$, la **perdida base elemento a
elemento** con la que se arman los terminos $\mathcal{L}_{fused}$ y $\mathcal{L}_{modality}$:

$$
\mathcal{L}_{fused} = \frac{1}{B}\sum_{i=1}^{B} \ell(\hat{y}_i,\, y_i),
\qquad
\mathcal{L}_{modality} = \frac{1}{B}\sum_{i=1}^{B}\sum_{d=1}^{M} \pi_{i,d}\,\ell(\hat{y}_{i,d},\, y_i)
$$

El resto de la funcion de costo ($\mathcal{L}_{agreement}$, $\widetilde{\mathcal{L}}_{KL}$,
$\widetilde{\mathcal{L}}_{rec}$, $\widetilde{\mathcal{L}}_{MI}$ y los pesos
$\gamma$/$\lambda$ por defecto) es **identica** en las tres — ver la seccion "Funcion de costo"
al inicio para la derivacion completa.

### Las tres candidatas

**1. `mse`** — referencia neutra; penaliza cuadraticamente, asi que la cola de `UITI_VANO`
domina el gradiente:

$$
\ell_{MSE}(\hat{y}, y) = (\hat{y} - y)^{2}
$$

**2. `huber`** — cuadratica cerca de cero, **lineal** en la cola (gradiente saturado en
$\delta$, asi que un residuo enorme no secuestra la actualizacion). `F.huber_loss` con
$\delta = 1.0$:

$$
\ell_{\delta}(\hat{y}, y) =
\begin{cases}
\dfrac{1}{2}\,(\hat{y} - y)^{2}, & |\hat{y} - y| \le \delta \\[2ex]
\delta\left(|\hat{y} - y| - \dfrac{1}{2}\delta\right), & |\hat{y} - y| > \delta
\end{cases}
$$

**3. `kernel_weighted_mse`** — MSE pesado por el **inverso de la densidad** del objetivo, para
subir el peso de las regiones raras de `y` (donde viven los vanos criticos y donde `mse` los
ahoga bajo la masa densa del rango medio):

$$
\ell_{KW}(\hat{y}, y) = w(y)\,(\hat{y} - y)^{2},
\qquad
w(y) = \frac{1}{\max\bigl(\hat{p}(y),\, \varepsilon\bigr)},\quad \varepsilon = 10^{-6}
$$

con $\hat{p}$ = KDE gaussiana ajustada **una sola vez** sobre `y_train` (regla de Scott),
evaluada en una malla fija de $K=512$ puntos e interpolada linealmente por lote.

### Como leer la tabla de resultados (importante)

> **No compares la columna de perdida entre filas.** Las tres $\ell$ viven en escalas
> distintas, y esa diferencia es de construccion, no de desempeno:
>
> - Como $y \in [0,1]$ tras `log1p`+MinMax, casi todo residuo cumple $|\hat{y}-y| \le \delta = 1$,
>   asi que `huber` opera casi siempre en su rama cuadratica y equivale a
>   $\tfrac{1}{2}\,\ell_{MSE}$ — **la mitad de la escala de `mse`**, por definicion.
> - Dentro de la perdida compuesta se llama a `compute_weights`, que devuelve el inverso crudo
>   de la densidad **sin renormalizar a media 1** (esa renormalizacion solo ocurre en
>   `KernelDensityWeightedMSELoss.forward`, que esta clase nunca invoca). `kernel_weighted_mse`
>   entra entonces en una escala varias veces mayor.
>
> Por eso la seleccion se hace por **`mae_original`** — MAE en la escala real de `UITI_VANO`,
> tras invertir MinMax y `expm1` — que es la unica metrica de la tabla invariante a la forma de
> la perdida. `best_valid_fused_loss` esta ahi para diagnostico dentro de una misma fila, no
> para rankear entre filas.

In [ ]:
kernel_loss_module = KernelDensityWeightedMSELoss.from_targets(y_train_scaled, n_grid=512)

loss_variants = {
    "mse": build_regression_loss("mse"),
    "huber": build_regression_loss("huber"),
    "kernel_weighted_mse": build_regression_loss("kernel_weighted_mse", kernel_loss_module=kernel_loss_module),
}

loss_comparison_rows = []
for name, loss_fn in loss_variants.items():
    print(f"--- Entrenando con base_loss={name} ---")
    _, _, metrics = train_regressor(
        loss_fn, max_epochs=LOSS_COMPARISON_MAX_EPOCHS, patience=LOSS_COMPARISON_PATIENCE, verbose=True,
    )
    loss_comparison_rows.append({"base_loss": name, **metrics})

loss_comparison_df = pd.DataFrame(loss_comparison_rows).set_index("base_loss")
BEST_BASE_LOSS = loss_comparison_df["mae_original"].idxmin()
print(loss_comparison_df)
print(f"\nMejor base_loss por mae_original: {BEST_BASE_LOSS}")

## 6. Busqueda de hiperparametros con Optuna

Mismo sampler/pruner que el baseline (`GPSampler` + `MedianPruner`, reutilizados via
`run_optuna_study` — nunca reimplementados), objetivo `mae_original` (minimizar).

El journal y el `.pkl` del estudio se escriben en `data/optuna/` con nombres sufijados por
`mode`, para no pisar los artefactos del baseline de `02.1_`. El journal es reanudable: volver a
correr esta celda con el mismo `mode` **continua** el estudio en vez de empezarlo de cero.

In [ ]:
OPTUNA_DIR = DATA_DIR / "optuna"
OPTUNA_DIR.mkdir(parents=True, exist_ok=True)

OPTUNA_JOURNAL_PATH = OPTUNA_DIR / f"mgcecdl_regression_local_{mode}.journal"
OPTUNA_STUDY_PATH = OPTUNA_DIR / f"mgcecdl_regression_local_{mode}.pkl"


def regression_objective(trial):
    params = {
        "hidden_dim": trial.suggest_categorical("hidden_dim", [64, 128, 192]),
        "embed_dim": trial.suggest_categorical("embed_dim", [32, 64, 96]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.25),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-4, log=True),
        "optimizer_type": trial.suggest_categorical("optimizer_type", ["adam", "adamw", "sgd", "rmsprop"]),
        "momentum": trial.suggest_float("momentum", 0.5, 0.95),
        "batch_size": trial.suggest_categorical("batch_size", [256, 512, 1024]),
        "gamma_sup": trial.suggest_float("gamma_sup", 1e-2, 1.0, log=True),
        "gamma_agr": trial.suggest_float("gamma_agr", 1e-2, 1.0, log=True),
        "gamma_reg": trial.suggest_float("gamma_reg", 1e-2, 1.0, log=True),
        "lambda_reconstruction": trial.suggest_float("lambda_reconstruction", 1e-2, 1.0, log=True),
        "lambda_mutual_information": trial.suggest_float("lambda_mutual_information", 1e-2, 1.0, log=True),
    }
    loss_fn = build_regression_loss(
        BEST_BASE_LOSS,
        gamma_sup=params["gamma_sup"], gamma_agr=params["gamma_agr"], gamma_reg=params["gamma_reg"],
        kernel_loss_module=kernel_loss_module if BEST_BASE_LOSS == "kernel_weighted_mse" else None,
        lambda_reconstruction=params["lambda_reconstruction"],
        lambda_mutual_information=params["lambda_mutual_information"],
    )
    _, _, metrics = train_regressor(
        loss_fn,
        hidden_dim=params["hidden_dim"], embed_dim=params["embed_dim"], dropout=params["dropout"],
        lr=params["learning_rate"], weight_decay=params["weight_decay"],
        optimizer_type=params["optimizer_type"], momentum=params["momentum"], batch_size=params["batch_size"],
        max_epochs=OPTUNA_MAX_EPOCHS, patience=OPTUNA_PATIENCE,
    )
    return metrics["mae_original"]


_search_t0 = _time.time()
regression_study = run_optuna_study(
    objective=regression_objective,
    study_name=f"mgcecdl_regression_local_{mode}",
    storage_path=OPTUNA_JOURNAL_PATH,
    n_trials=OPTUNA_N_TRIALS,
    seed=RANDOM_STATE,
    direction="minimize",
)
_search_elapsed = _time.time() - _search_t0
guardar_estudio_optuna(regression_study, OPTUNA_STUDY_PATH)

print(f"Optuna: {OPTUNA_N_TRIALS} trials en {_search_elapsed:.1f}s "
      f"(promedio {_search_elapsed / max(OPTUNA_N_TRIALS, 1):.1f}s/trial)")
print(f"Mejor mae_original: {regression_study.best_value:.4f}")
print(f"Mejores hiperparametros: {json.dumps(regression_study.best_params, indent=2)}")
print(f"Journal: {OPTUNA_JOURNAL_PATH}")
print(f"Estudio: {OPTUNA_STUDY_PATH}")

### Graficos de diagnostico de Optuna

Ventaja de correr en local frente a Kaggle: los PNG quedan versionables en
`reports/interpretability/figures/`. Se usa `optuna.visualization.matplotlib` para no depender
de `kaleido`/navegador. Con muy pocos trials (modo `smoke`) el calculo de importancias puede no
converger; en ese caso se avisa y se sigue, sin abortar el cuaderno.

**Barra de color del grafico de contornos.** Optuna la dibuja vertical a la derecha, quitandole
ancho a los paneles (que en una grilla 3x3 ya vienen apretados). `contour_with_horizontal_colorbar`
la retira, devuelve ese ancho a la grilla y la redibuja **horizontal debajo**, midiendo antes la
extension real de los paneles (`get_tightbbox`, ya renderizados) para garantizar que no se
traslape con ellos ni con sus etiquetas.

In [ ]:
import matplotlib.pyplot as plt
import optuna.visualization.matplotlib as optuna_mpl


# plot_contour de Optuna coloca la barra de color VERTICAL a la derecha
# (`fig.colorbar(cs, ax=axs)`), robandole ancho a todos los paneles. Aqui se retira esa barra,
# la grilla recupera el ancho completo, y la barra se vuelve a dibujar horizontal por debajo.
def contour_with_horizontal_colorbar(study, params, figsize=(9.5, 9.0)):
    contour_axes = optuna_mpl.plot_contour(study, params=params)
    grid_axes = list(np.atleast_1d(contour_axes).ravel())
    fig = grid_axes[0].figure
    grid_ids = {id(ax) for ax in grid_axes}

    # La barra de Optuna es un Axes extra, fuera de la grilla, con su Colorbar colgado en
    # `_colorbar`; de ahi se recuperan el mappable y la etiqueta antes de retirarla.
    mappable = None
    colorbar_label = ""
    for ax in list(fig.axes):
        colorbar = getattr(ax, "_colorbar", None)
        if id(ax) not in grid_ids and colorbar is not None:
            mappable = colorbar.mappable
            colorbar_label = ax.get_ylabel() or ax.get_xlabel()
            colorbar.remove()

    fig.set_size_inches(*figsize)
    # La grilla recupera el ancho de la barra vertical; el margen inferior reserva el sitio
    # para la barra horizontal. Despues de esto NO se debe llamar a tight_layout(): recalcularia
    # las posiciones y volveria a pegar la barra a los paneles.
    # `left` es holgado a proposito: los nombres de hiperparametro largos (p. ej.
    # "optimizer_type") mas sus etiquetas de tick categoricas ("rmsprop") se salen del lienzo
    # con un margen apretado.
    left_margin, right_margin = 0.12, 0.98
    fig.subplots_adjust(
        left=left_margin, right=right_margin, bottom=0.20, top=0.93, wspace=0.28, hspace=0.28,
    )

    if mappable is not None:
        # Se renderiza primero para medir la extension REAL de la grilla (ejes + etiquetas de
        # tick + titulos de eje) y colocar la barra estrictamente por debajo. Medir en vez de
        # asumir es lo que garantiza que no haya traslape con ejes log o etiquetas largas.
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        to_figure_coords = fig.transFigure.inverted()
        grid_bottom = min(
            to_figure_coords.transform(ax.get_tightbbox(renderer))[0][1] for ax in grid_axes
        )
        bar_height = 0.016
        gap = 0.035
        # El piso de 0.075 deja aire para la etiqueta de la barra, que se dibuja por debajo.
        bar_y = max(grid_bottom - gap - bar_height, 0.075)
        cax = fig.add_axes([left_margin, bar_y, right_margin - left_margin, bar_height])
        horizontal_colorbar = fig.colorbar(mappable, cax=cax, orientation="horizontal")
        horizontal_colorbar.set_label(colorbar_label)
    return fig


completed_trials = [t for t in regression_study.trials if t.state == optuna.trial.TrialState.COMPLETE]
if len(completed_trials) < 2:
    print(f"Solo {len(completed_trials)} trial(s) completado(s): se omiten los graficos de diagnostico.")
else:
    try:
        importance_ax = optuna_mpl.plot_param_importances(regression_study)
        importance_fig = importance_ax.figure
        importance_fig.tight_layout()
        IMPORTANCE_FIGURE_PATH = FIGURES_DIR / f"mgcecdl_regression_local_{mode}_param_importances.png"
        importance_fig.savefig(IMPORTANCE_FIGURE_PATH, dpi=150)
        print("Importancias:", IMPORTANCE_FIGURE_PATH)

        importances = optuna.importance.get_param_importances(regression_study)
        top_params = list(importances)[:3]
        if len(top_params) >= 2:
            contour_fig = contour_with_horizontal_colorbar(regression_study, top_params)
            CONTOUR_FIGURE_PATH = FIGURES_DIR / f"mgcecdl_regression_local_{mode}_contour.png"
            contour_fig.savefig(CONTOUR_FIGURE_PATH, dpi=150)
            print("Contorno:", CONTOUR_FIGURE_PATH)
        plt.show()
    except Exception as exc:  # graficos de diagnostico: nunca deben tumbar la corrida
        print(f"No se pudieron generar los graficos de diagnostico de Optuna: {exc!r}")

## 7. Reentrenamiento final (mejor forma de perdida + mejores hiperparametros)

In [ ]:
best_params = regression_study.best_params
final_loss_fn = build_regression_loss(
    BEST_BASE_LOSS,
    gamma_sup=best_params["gamma_sup"], gamma_agr=best_params["gamma_agr"], gamma_reg=best_params["gamma_reg"],
    kernel_loss_module=kernel_loss_module if BEST_BASE_LOSS == "kernel_weighted_mse" else None,
    lambda_reconstruction=best_params["lambda_reconstruction"],
    lambda_mutual_information=best_params["lambda_mutual_information"],
)

regressor, final_history, final_metrics = train_regressor(
    final_loss_fn,
    hidden_dim=best_params["hidden_dim"], embed_dim=best_params["embed_dim"], dropout=best_params["dropout"],
    lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"],
    optimizer_type=best_params["optimizer_type"], momentum=best_params["momentum"],
    batch_size=best_params["batch_size"],
    max_epochs=FINAL_MAX_EPOCHS, patience=FINAL_PATIENCE, verbose=True,
)

print("\n===== Modelo final (mejores hiperparametros + mejor forma de perdida) =====")
for key, value in final_metrics.items():
    print(f"{key}: {value}")

### Curvas de perdida del reentrenamiento final

Dos paneles lado a lado, ambos sobre las epochs del reentrenamiento final:

**Izquierda -- perdida total.** `total_loss` en entrenamiento y en validacion. Es la cantidad
que el optimizador minimiza. La linea vertical marca el epoch del checkpoint restaurado (el de
menor `valid_fused_loss`, que es el criterio de early stopping y **no** necesariamente el de
menor `total_loss`: si ambas marcas no coinciden, es que los terminos auxiliares seguian
mejorando mientras el ajuste supervisado ya empeoraba).

**Derecha -- desglose por termino.** Cada termino **ya multiplicado por su peso**
($\gamma_{sup}$, $\gamma_{agr}$, $\gamma_{reg}$, $\lambda_{rec}$, $\lambda_{MI}$), de modo que
las seis curvas suman exactamente la curva total de la izquierda. Esto es lo que hace legible
el grafico: un termino con valor crudo grande pero peso $0.01$ no manda en la optimizacion, y
graficar los valores crudos lo haria parecer dominante.

Escala **logaritmica** en el eje y, porque los terminos difieren en varios ordenes de magnitud.
Se grafica el desglose de **entrenamiento**, que es el que genera los gradientes.

**Que buscar.** Si una sola curva queda pegada al total y las demas viven decadas por debajo,
la optimizacion esta gobernada por ese termino y los pesos de los otros son decorativos —
justo el tipo de hallazgo que justificaria reajustar los rangos que Optuna explora.

In [ ]:
from matplotlib.ticker import MaxNLocator

final_history_df = pd.DataFrame(final_history).set_index("epoch")
term_weights = loss_term_weights(final_loss_fn)

fig, (ax_total, ax_terms) = plt.subplots(1, 2, figsize=(15, 5.5))

# --- Panel izquierdo: perdida total, train vs valid ---
ax_total.plot(final_history_df.index, final_history_df["train_total_loss"],
              marker="o", markersize=3, label="train")
ax_total.plot(final_history_df.index, final_history_df["valid_total_loss"],
              marker="o", markersize=3, label="valid")
best_epoch_final = final_metrics["best_epoch"]
ax_total.axvline(best_epoch_final, linestyle="--", color="grey",
                 label=f"checkpoint restaurado (epoch {best_epoch_final})")
ax_total.set_xlabel("Epoch")
ax_total.set_ylabel("total_loss")
ax_total.set_title("Perdida total por epoch")
ax_total.legend(fontsize=9)

# --- Panel derecho: contribucion ponderada de cada termino (suman el total) ---
term_colors = plt.get_cmap("tab10").colors
for position, (term, weight) in enumerate(term_weights.items()):
    contribution = final_history_df[f"train_{term}"] * weight
    ax_terms.plot(
        final_history_df.index, contribution, marker="o", markersize=2.5,
        color=term_colors[position % len(term_colors)],
        label=f"{term.replace('_loss', '')} (x{weight:.3g})",
    )
ax_terms.plot(final_history_df.index, final_history_df["train_total_loss"],
              color="black", linewidth=1.6, linestyle=":", label="total (suma)")
ax_terms.axvline(best_epoch_final, linestyle="--", color="grey")
ax_terms.set_yscale("log")  # los terminos difieren en varios ordenes de magnitud
ax_terms.set_xlabel("Epoch")
ax_terms.set_ylabel("Contribucion ponderada al total (escala log)")
ax_terms.set_title("Desglose por termino (entrenamiento, ya ponderado)")
ax_terms.legend(fontsize=8, ncol=2)

for ax in (ax_total, ax_terms):
    # Los epochs son enteros: sin esto matplotlib rotula 1.25, 1.50, ... cuando hay pocos.
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

fig.suptitle(
    f"Reentrenamiento final -- base_loss={BEST_BASE_LOSS}, {len(final_history_df)} epochs, "
    f"device={DEVICE}, mode={mode}",
    fontsize=12,
)
fig.tight_layout()
LOSS_CURVES_FIGURE_PATH = FIGURES_DIR / f"mgcecdl_regression_local_{mode}_loss_curves.png"
fig.savefig(LOSS_CURVES_FIGURE_PATH, dpi=150)
print("Figura:", LOSS_CURVES_FIGURE_PATH)
plt.show()

# Verificacion numerica del desglose: la suma ponderada de los terminos debe reproducir el
# total registrado. Si no cuadra, el grafico de la derecha estaria mintiendo.
reconstructed_total = sum(
    final_history_df[f"train_{term}"] * weight for term, weight in term_weights.items()
)
max_gap = float((reconstructed_total - final_history_df["train_total_loss"]).abs().max())
print(f"Maxima discrepancia entre la suma de terminos ponderados y total_loss: {max_gap:.3e}")
assert max_gap < 1e-4, "El desglose no reproduce total_loss: revisar TRACKED_LOSS_KEYS/pesos."

print("\nContribucion media de cada termino al total (entrenamiento, ultimo epoch):")
last_epoch_row = final_history_df.iloc[-1]
for term, weight in sorted(
    term_weights.items(), key=lambda kv: last_epoch_row[f"train_{kv[0]}"] * kv[1], reverse=True
):
    contribution = last_epoch_row[f"train_{term}"] * weight
    share = 100.0 * contribution / last_epoch_row["train_total_loss"]
    print(f"  {term:24s} crudo={last_epoch_row[f'train_{term}']:.5f} "
          f"x peso={weight:.4g} -> {contribution:.5f} ({share:5.1f}% del total)")

# Diagnostico de saturacion. `reconstruction_loss` y `mutual_information_loss` salen de un
# clip(0, 1) en el modelo; pegado al techo, el gradiente de ese termino es EXACTAMENTE cero:
# sigue inflando el valor de la perdida pero ya no ensena nada. Es una patologia silenciosa
# que el numero del total no delata, asi que se avisa explicitamente.
SATURATION_TOLERANCE = 1e-6
for clipped_term in ("reconstruction_loss", "mutual_information_loss"):
    series = final_history_df[f"train_{clipped_term}"]
    saturated_epochs = int((series >= 1.0 - SATURATION_TOLERANCE).sum())
    if saturated_epochs:
        print(
            f"\nAVISO: '{clipped_term}' quedo saturado en 1.0 durante {saturated_epochs}/"
            f"{len(series)} epochs. Ese clip anula su gradiente: el termino aporta valor a la "
            "perdida pero no informacion al optimizador. Si persiste en mode='full', el peso "
            "correspondiente esta comprando ruido y conviene revisar su rango en Optuna."
        )

## 8. Extraccion de embeddings (todos los eventos, modelo final)

Se concatenan los embeddings de todas las modalidades por evento y se promedian por
`CIRCUITO` + `FID_VANO`: la unidad de analisis del agrupamiento es el vano.

In [ ]:
regressor.eval()
full_dataset = TensorDataset(torch.tensor(X_full_scaled))
full_loader = DataLoader(full_dataset, batch_size=2048, shuffle=False)

event_embeddings_chunks = []
with torch.no_grad():
    for (xb,) in full_loader:
        xb = xb.to(DEVICE)
        out = regressor(xb)
        concatenated = torch.cat(out["embeddings"], dim=1)
        event_embeddings_chunks.append(concatenated.cpu().numpy())
event_embeddings = np.vstack(event_embeddings_chunks)

embedding_columns = [f"embed_{i}" for i in range(event_embeddings.shape[1])]
event_embeddings_df = pd.DataFrame(event_embeddings, columns=embedding_columns)
event_embeddings_df["CIRCUITO"] = df_identidad["CIRCUITO"].values
event_embeddings_df["FID_VANO"] = df_identidad["FID_VANO"].astype(str).values

vano_embeddings_df = (
    event_embeddings_df.groupby(["CIRCUITO", "FID_VANO"])[embedding_columns].mean().reset_index()
)
print("event_embeddings:", event_embeddings.shape)
print("vano_embeddings_df:", vano_embeddings_df.shape)

## 9. K-Means + silueta sobre los embeddings (`K=2..8`)

In [ ]:
embedding_matrix = StandardScaler().fit_transform(vano_embeddings_df[embedding_columns].values)

n_vanos = embedding_matrix.shape[0]
k_upper_bound = min(9, n_vanos)  # KMeans exige n_samples >= n_clusters; protege submuestras smoke
K_RANGE = range(2, k_upper_bound)
if len(K_RANGE) == 0:
    raise RuntimeError(
        f"Solo hay {n_vanos} vanos distintos en esta corrida -- sube SMOKE_EVENT_SUBSAMPLE "
        "o usa mode='full'."
    )

embedding_cluster_rows = []
embedding_labels_by_k = {}
for k in K_RANGE:
    kmeans_embed = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans_embed.fit_predict(embedding_matrix)
    silhouette = silhouette_score(embedding_matrix, labels)
    embedding_cluster_rows.append({"k": k, "inertia": kmeans_embed.inertia_, "silhouette": silhouette})
    embedding_labels_by_k[k] = labels

embedding_cluster_df = pd.DataFrame(embedding_cluster_rows).set_index("k")
best_k_embeddings = int(embedding_cluster_df["silhouette"].idxmax())
print(embedding_cluster_df)
print(f"K recomendado por silueta (maximo global): {best_k_embeddings}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(embedding_cluster_df.index, embedding_cluster_df["inertia"], marker="o")
axes[0].set_xlabel("K")
axes[0].set_ylabel("Inercia")
axes[0].set_title("Codo (inercia vs. K)")
axes[1].plot(embedding_cluster_df.index, embedding_cluster_df["silhouette"], marker="o", color="tab:orange")
axes[1].axvline(best_k_embeddings, linestyle="--", color="grey")
axes[1].set_xlabel("K")
axes[1].set_ylabel("Silueta")
axes[1].set_title(f"Silueta vs. K (maximo en K={best_k_embeddings})")
fig.suptitle("Agrupamiento de vanos sobre embeddings del MGCECDLRegressor")
fig.tight_layout()
ELBOW_FIGURE_PATH = FIGURES_DIR / f"mgcecdl_regression_local_{mode}_elbow_silhouette.png"
fig.savefig(ELBOW_FIGURE_PATH, dpi=150)
print("Figura:", ELBOW_FIGURE_PATH)
plt.show()

## 10. Triangulacion contra la Parte A (K-Means sobre features crudas, `10_uiti_vano_kmeans.ipynb`)

Se recalcula de forma independiente desde el CSV (misma fuente a nivel de evento que la entrada
del modelo) y se une por `CIRCUITO` + `FID_VANO`. `K=4` se triangula contra la Parte A
independientemente del `K` que elija la silueta arriba, para quedar comparable con el
`ARI(K=4)=0.1115` del baseline.

In [ ]:
raw_events_df = pd.read_csv(DATASET_PATH, usecols=["CIRCUITO", "FID_VANO", "UITI_VANO"])
raw_events_df["FID_VANO"] = (
    raw_events_df["FID_VANO"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
)
raw_events_df["UITI_VANO"] = pd.to_numeric(raw_events_df["UITI_VANO"], errors="coerce").fillna(0.0)

vano_table_partA = (
    raw_events_df.groupby(["CIRCUITO", "FID_VANO"])
    .agg(uiti_acumulado=("UITI_VANO", "sum"), num_eventos=("UITI_VANO", "count"))
    .reset_index()
)
log_features_partA = np.column_stack([
    np.log10(vano_table_partA["num_eventos"]), np.log10(vano_table_partA["uiti_acumulado"]),
])
log_features_partA_scaled = MinMaxScaler().fit_transform(log_features_partA)

PART_A_SILHOUETTE_K = 2
PART_A_PRODUCTION_K = 4
vano_table_partA["cluster_partA_silhouette_k"] = KMeans(
    n_clusters=PART_A_SILHOUETTE_K, random_state=42, n_init=10
).fit_predict(log_features_partA_scaled)
vano_table_partA["cluster_partA_k4"] = KMeans(
    n_clusters=PART_A_PRODUCTION_K, random_state=42, n_init=10
).fit_predict(log_features_partA_scaled)

vano_embeddings_df["cluster_partB_silhouette_k"] = embedding_labels_by_k[best_k_embeddings]
vano_embeddings_df["cluster_partB_k4"] = embedding_labels_by_k.get(4, embedding_labels_by_k[best_k_embeddings])

triangulation_df = vano_table_partA.merge(
    vano_embeddings_df[["CIRCUITO", "FID_VANO", "cluster_partB_silhouette_k", "cluster_partB_k4"]],
    on=["CIRCUITO", "FID_VANO"], how="inner",
)
print("Vanos en Parte A:", len(vano_table_partA))
print("Vanos en Parte B (embeddings de esta corrida):", len(vano_embeddings_df))
print("Vanos con ambos agrupamientos (join):", len(triangulation_df))

ari_own_best_k = adjusted_rand_score(
    triangulation_df["cluster_partA_silhouette_k"], triangulation_df["cluster_partB_silhouette_k"]
)
ari_matched_k4 = adjusted_rand_score(triangulation_df["cluster_partA_k4"], triangulation_df["cluster_partB_k4"])
print(f"ARI (K propio por lado -- Parte A K={PART_A_SILHOUETTE_K}, Parte B K={best_k_embeddings}): {ari_own_best_k:.4f}")
print(f"ARI (K=4 fijo en ambos lados): {ari_matched_k4:.4f}")

## 11. Proyeccion UMAP 2D de los embeddings

Los embeddings por vano viven en $M \times$ `embed_dim` dimensiones — imposible de mirar
directamente. UMAP los proyecta a 2D **preservando estructura local**, para poder ver si los
clusters de la seccion 9 son grupos reales y separados o cortes arbitrarios de una nube continua.

Se proyecta exactamente la misma `embedding_matrix` que alimento al K-Means (embeddings por
vano, estandarizados), asi que lo que se ve es la geometria sobre la que se decidio el
agrupamiento, no otra distinta.

**Tres paneles en linea, misma proyeccion, distinto color:**

1. **Cluster K-Means** con el $K$ elegido por silueta en la seccion 9.
2. **UITI acumulado del vano** — suma de `UITI_VANO` sobre **todos** los eventos del vano.
3. **Numero de eventos del vano** — conteo de filas del vano.

Los dos ultimos se toman del **dataset completo** (`vano_table_partA`, agregado directamente
desde `Indicadores_vano_v3.csv` en la seccion 10), no de los eventos que entraron al modelo:
en `mode='smoke'` el modelo ve una submuestra, pero el UITI acumulado y el conteo de eventos
de cada vano siguen siendo los reales.

Ambos se colorean en **escala log10** porque su distribucion es de cola larga (misma decision
que `10_uiti_vano_kmeans.ipynb`); en escala lineal, un punado de vanos extremos aplasta todo
el resto del gradiente a un solo color. Ademas se recorta el rango de color a los percentiles
2-98, porque unos pocos vanos de UITI casi nulo estiraban la escala varias decadas.

**Los ticks de las barras van en unidades originales**, no en logaritmo: la suma de UITI y el
conteo de eventos tal como se leen en el dataset (1, 2, 5, 10, 20, 50, ...). El color sigue
mapeando el logaritmo — que es lo que hace legible el gradiente — pero un tick que dijera
`2.5` no le sirve a nadie para dimensionar un vano.

**Como leerlo.** Si los colores de los paneles 2 y 3 se organizan en gradientes que siguen a
los clusters del panel 1, los embeddings capturaron la severidad/frecuencia del vano. Si se ven
mezclados dentro de cada cluster, los embeddings agruparon por otra cosa — que es
justamente la hipotesis que el `ARI` bajo de la seccion 10 ya sugiere.

In [ ]:
import umap
from matplotlib.colors import BoundaryNorm, ListedColormap

n_vanos_embed = embedding_matrix.shape[0]
if n_vanos_embed < 5:
    raise RuntimeError(
        f"Solo {n_vanos_embed} vanos: UMAP necesita mas puntos. Sube SMOKE_EVENT_SUBSAMPLE "
        "o usa mode='full'."
    )

# n_neighbors debe ser < n_samples; con el default (15) una submuestra smoke pequena reventaria.
umap_n_neighbors = int(min(15, n_vanos_embed - 1))
_umap_t0 = _time.time()
umap_reducer = umap.UMAP(
    n_components=2,
    n_neighbors=umap_n_neighbors,
    min_dist=0.1,
    metric="euclidean",
    random_state=RANDOM_STATE,  # reproducible; desactiva el paralelismo de UMAP a proposito
)
umap_2d = umap_reducer.fit_transform(embedding_matrix)
print(f"UMAP: {n_vanos_embed} vanos -> 2D en {_time.time() - _umap_t0:.1f}s "
      f"(n_neighbors={umap_n_neighbors}, min_dist=0.1)")

# Agregados del DATASET COMPLETO. El merge es left sobre vano_embeddings_df para conservar el
# orden de las filas, que es el que corresponde a las filas de embedding_matrix / umap_2d.
umap_frame = vano_embeddings_df[["CIRCUITO", "FID_VANO", "cluster_partB_silhouette_k"]].merge(
    vano_table_partA[["CIRCUITO", "FID_VANO", "uiti_acumulado", "num_eventos"]],
    on=["CIRCUITO", "FID_VANO"], how="left",
)
assert len(umap_frame) == n_vanos_embed, "El merge altero el numero de filas: se perdio la alineacion con umap_2d."

has_aggregates = umap_frame["uiti_acumulado"].notna().to_numpy()
if not has_aggregates.all():
    print(f"AVISO: {(~has_aggregates).sum()} vano(s) sin agregados en el CSV completo; "
          "se omiten en los paneles 2 y 3.")

In [ ]:
def add_horizontal_colorbar(fig, ax, mappable, label, ticks=None, tick_labels=None):
    # Misma decision que en los diagnosticos de Optuna: barra horizontal DEBAJO del panel,
    # posicionada midiendo la extension real ya renderizada para que no la toque.
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    to_figure_coords = fig.transFigure.inverted()
    panel_box = ax.get_position()
    panel_bottom = to_figure_coords.transform(ax.get_tightbbox(renderer))[0][1]
    bar_height = 0.022
    bar_y = max(panel_bottom - 0.055 - bar_height, 0.02)
    cax = fig.add_axes([panel_box.x0, bar_y, panel_box.width, bar_height])
    colorbar = fig.colorbar(mappable, cax=cax, orientation="horizontal")
    if ticks is not None:
        colorbar.set_ticks(ticks)
        if tick_labels is not None:
            colorbar.set_ticklabels(tick_labels)
    colorbar.set_label(label)
    return colorbar


cluster_values = umap_frame["cluster_partB_silhouette_k"].to_numpy()
cluster_ids = np.unique(cluster_values)
point_size = 6.0 if n_vanos_embed > 5000 else 12.0

fig, axes = plt.subplots(1, 3, figsize=(16.5, 6.0))
fig.subplots_adjust(left=0.05, right=0.98, bottom=0.26, top=0.88, wspace=0.18)

# Panel 1 -- clusters (categorico): colormap discreto + BoundaryNorm para que la barra muestre
# un bloque por cluster y no un degradado continuo, que sugeriria un orden inexistente.
cluster_cmap = ListedColormap(plt.get_cmap("tab10").colors[: len(cluster_ids)])
cluster_norm = BoundaryNorm(np.arange(len(cluster_ids) + 1) - 0.5, len(cluster_ids))
scatter_clusters = axes[0].scatter(
    umap_2d[:, 0], umap_2d[:, 1], c=cluster_values, cmap=cluster_cmap, norm=cluster_norm,
    s=point_size, alpha=0.75, linewidths=0,
)
axes[0].set_title(f"Clusters K-Means (K={best_k_embeddings}, elegido por silueta)")

# Limites de color robustos (percentiles 2-98): un punado de vanos con UITI casi nulo estira
# la escala log varias decadas y aplasta a un solo tono la zona donde vive la mayoria de los
# datos. Los valores fuera del rango se siguen dibujando, saturados en el color del extremo.
def robust_color_limits(values, low=2.0, high=98.0):
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return None, None
    vmin, vmax = np.percentile(finite, [low, high])
    if vmin == vmax:  # constante: dejar que matplotlib elija
        return None, None
    return float(vmin), float(vmax)


# El color mapea log10(valor), pero un tick que diga "2.5" no se lee como nada util. Estas
# posiciones siguen estando en escala log (que es donde vive el color), pero se ETIQUETAN con
# el valor original: numero de eventos y suma de UITI tal como se cuentan en el dataset.
def log_ticks_in_original_units(log_vmin, log_vmax, max_ticks=8):
    if log_vmin is None or log_vmax is None:
        return None, None
    # Malla 1-2-5 por decada: en rangos de 1-2 decadas (el caso de num_eventos) las decadas
    # solas dejarian dos o tres ticks en toda la barra.
    candidates = sorted(
        {mantissa * (10.0 ** exponent)
         for exponent in range(int(np.floor(log_vmin)), int(np.ceil(log_vmax)) + 1)
         for mantissa in (1, 2, 5)}
    )
    inside = [v for v in candidates if log_vmin <= np.log10(v) <= log_vmax]
    if not inside:  # rango mas estrecho que un tick: usar los propios extremos
        inside = [10.0 ** log_vmin, 10.0 ** log_vmax]
    if len(inside) > max_ticks:
        inside = inside[:: int(np.ceil(len(inside) / max_ticks))]
    labels = [f"{v:,.0f}" if v >= 1 else f"{v:g}" for v in inside]
    return np.log10(inside), labels


log_uiti = np.log10(umap_frame.loc[has_aggregates, "uiti_acumulado"].clip(lower=1e-6).to_numpy())
uiti_vmin, uiti_vmax = robust_color_limits(log_uiti)
scatter_uiti = axes[1].scatter(
    umap_2d[has_aggregates, 0], umap_2d[has_aggregates, 1],
    c=log_uiti, cmap="viridis", vmin=uiti_vmin, vmax=uiti_vmax,
    s=point_size, alpha=0.75, linewidths=0,
)
axes[1].set_title("UITI acumulado por vano (dataset completo)")

log_eventos = np.log10(umap_frame.loc[has_aggregates, "num_eventos"].clip(lower=1).to_numpy())
eventos_vmin, eventos_vmax = robust_color_limits(log_eventos)
scatter_eventos = axes[2].scatter(
    umap_2d[has_aggregates, 0], umap_2d[has_aggregates, 1],
    c=log_eventos, cmap="magma", vmin=eventos_vmin, vmax=eventos_vmax,
    s=point_size, alpha=0.75, linewidths=0,
)
axes[2].set_title("Numero de eventos por vano (dataset completo)")

for ax in axes:
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    # Los ejes de UMAP no tienen unidades interpretables: solo importa la geometria relativa.
    ax.set_xticklabels([])
    ax.set_yticklabels([])

fig.suptitle(
    f"Proyeccion UMAP 2D de los embeddings del MGCECDLRegressor por vano "
    f"(n={n_vanos_embed}, mode={mode})",
    fontsize=13,
)

uiti_ticks, uiti_tick_labels = log_ticks_in_original_units(uiti_vmin, uiti_vmax)
eventos_ticks, eventos_tick_labels = log_ticks_in_original_units(eventos_vmin, eventos_vmax)

add_horizontal_colorbar(fig, axes[0], scatter_clusters, "Cluster",
                        ticks=cluster_ids, tick_labels=[str(c) for c in cluster_ids])
add_horizontal_colorbar(fig, axes[1], scatter_uiti,
                        "Suma de UITI del vano  (color log, p2-p98)",
                        ticks=uiti_ticks, tick_labels=uiti_tick_labels)
add_horizontal_colorbar(fig, axes[2], scatter_eventos,
                        "Numero de eventos del vano  (color log, p2-p98)",
                        ticks=eventos_ticks, tick_labels=eventos_tick_labels)

UMAP_FIGURE_PATH = FIGURES_DIR / f"mgcecdl_regression_local_{mode}_umap_clusters.png"
fig.savefig(UMAP_FIGURE_PATH, dpi=150)
print("Figura:", UMAP_FIGURE_PATH)
plt.show()

In [ ]:
# Lectura cuantitativa que acompana a la figura: si los embeddings capturaron severidad y
# frecuencia, estas medianas deberian separarse claramente entre clusters.
cluster_profile = (
    umap_frame.loc[has_aggregates]
    .groupby("cluster_partB_silhouette_k")
    .agg(
        n_vanos=("FID_VANO", "count"),
        uiti_acumulado_mediana=("uiti_acumulado", "median"),
        uiti_acumulado_p90=("uiti_acumulado", lambda s: s.quantile(0.90)),
        num_eventos_mediana=("num_eventos", "median"),
    )
    .round(2)
)
print("Perfil de cada cluster (agregados del dataset completo):")
print(cluster_profile)

## 12. Reporte: `mae_original` frente al baseline local fijado

`mae_original` es la metrica primaria de exito/comparacion; los $R^2$ y los ARI son
diagnosticos secundarios y **nunca la reemplazan**. Las constantes de abajo estan fijadas en
`.claude/skills/experimento-kaggle/references/uiti-vano-regression-baseline.md` (fuente de
verdad).

**Sobre los $R^2$ del baseline.** Los valores fijados alli son $R^2$ **planos**, medidos antes
de que este cuaderno agregara el ajustado. Para no comparar peras con manzanas, el reporte
hace dos cosas: contrasta el $R^2$ plano de la corrida contra el plano del baseline
(comparacion directa, siempre valida), y ademas convierte el baseline a ajustado usando el
$n$ y $p$ de **esta** corrida. Esa conversion solo es legitima en `mode='full'`, donde la
particion y el numero de predictores son identicos a los del baseline por construccion; en
`mode='smoke'` la submuestra cambia $n$, asi que se marca como no comparable.

In [ ]:
BASELINE_MAE_ORIGINAL = 126.402  # fijado en references/uiti-vano-regression-baseline.md
BASELINE_R2_ORIGINAL = -0.027
BASELINE_R2_TRANSFORMED = 0.284
BASELINE_ARI_AUTO_K = 0.0000
BASELINE_ARI_K4 = 0.1115

run_mae_original = final_metrics["mae_original"]
delta = run_mae_original - BASELINE_MAE_ORIGINAL
improved = run_mae_original < BASELINE_MAE_ORIGINAL

print(f"mode: {mode}")
if mode == "smoke":
    print("AVISO: modo 'smoke' -- presupuesto y submuestra de juguete. "
          "Estos numeros NO son comparables con el baseline; solo prueban el cableado.")
print(f"mae_original de la corrida: {run_mae_original:.4f}")
print(f"mae_original del baseline:  {BASELINE_MAE_ORIGINAL:.4f}")
print(f"delta (corrida - baseline): {delta:+.4f}  ({'MEJORA (menor)' if improved else 'no mejora'})")
print()

n_valid_final = final_metrics["n_valid"]
p_final = final_metrics["n_predictors"]
adjustment_factor = (n_valid_final - 1) / (n_valid_final - p_final - 1)
print(f"R2 ajustado: n_valid={n_valid_final}, p={p_final}, "
      f"factor (n-1)/(n-p-1)={adjustment_factor:.4f}")
print("  r2_original      plano / ajustado: "
      f"{final_metrics['r2_original']:.4f} / {final_metrics['r2_original_adj']:.4f}")
print("  r2_transformed   plano / ajustado: "
      f"{final_metrics['r2_transformed']:.4f} / {final_metrics['r2_transformed_adj']:.4f}")
print()
print("Diagnosticos secundarios (esta corrida vs. baseline):")
print(f"  r2_original    (plano): {final_metrics['r2_original']:.4f}  vs. {BASELINE_R2_ORIGINAL:.4f}")
print(f"  r2_transformed (plano): {final_metrics['r2_transformed']:.4f}  vs. {BASELINE_R2_TRANSFORMED:.4f}")
if mode == "full":
    # Particion y numero de predictores identicos al baseline por construccion, asi que el
    # baseline se puede reexpresar como ajustado con el n/p de esta corrida.
    baseline_r2_original_adj = adjusted_r2(BASELINE_R2_ORIGINAL, n_valid_final, p_final)
    baseline_r2_transformed_adj = adjusted_r2(BASELINE_R2_TRANSFORMED, n_valid_final, p_final)
    print(f"  r2_original    (ajust): {final_metrics['r2_original_adj']:.4f}  vs. {baseline_r2_original_adj:.4f}")
    print(f"  r2_transformed (ajust): {final_metrics['r2_transformed_adj']:.4f}  vs. {baseline_r2_transformed_adj:.4f}")
else:
    print("  (ajustado del baseline omitido: en mode='smoke' la submuestra cambia n, "
          "la conversion no seria comparable)")
print(f"  ARI (K auto):           {ari_own_best_k:.4f}  vs. {BASELINE_ARI_AUTO_K:.4f}")
print(f"  ARI (K=4):              {ari_matched_k4:.4f}  vs. {BASELINE_ARI_K4:.4f}")
print()
print(f"Optuna: {OPTUNA_N_TRIALS} trials @ {OPTUNA_MAX_EPOCHS} epochs | "
      f"forma de perdida seleccionada: {BEST_BASE_LOSS} | "
      f"reentrenamiento final: {FINAL_MAX_EPOCHS} epochs | dispositivo: {DEVICE}")